# Hallucination detection for OpenAI responses with styxx

This notebook shows how to wrap any OpenAI call with runtime hallucination detection using [styxx](https://github.com/fathom-lab/styxx) — a 9-signal hallucination detector cross-validated across 8 public benchmarks.

**What you'll do:**
1. `pip install styxx[nli]`
2. Wrap any function that calls the OpenAI API with `@trust`
3. See the detector pass-through a correct answer and intercept a hallucinated one
4. Inspect the per-signal verdict (risk, action, signal table)

**Why this matters for production OpenAI workloads:**
- Cross-validated AUCs on public benchmarks:

| Benchmark | AUC |
|---|---|
| HaluEval-QA | 0.998 |
| TruthfulQA | 0.994 |
| HaluBench-RAGTruth (RAG faithfulness) | 0.807 |
| HaluBench-PubMedQA | 0.719 |
| HaluEval-Dialog | 0.676 |
| HaluEval-Summarization | 0.643 |
| HaluBench-FinanceBench | 0.492 (declared failure mode) |
| HaluBench-DROP | 0.424 (declared failure mode) |

- Two failure modes published openly so you know where the detector **will not** help (extractive-span reading-comp errors on DROP-like tasks and financial arithmetic on FinanceBench-like tasks).
- Local-first: runs offline on CPU (~400ms) or CUDA (~30ms). No extra API key required.
- MIT on code, CC-BY-4.0 on calibrated weights.

Full manifesto + paper: https://fathom.darkflobi.com/cognometry · DOI: [10.5281/zenodo.19703527](https://doi.org/10.5281/zenodo.19703527).

---

## 1. Install

In [ ]:
%pip install -q styxx[nli] openai

## 2. Set your OpenAI key

Either export `OPENAI_API_KEY` in your shell or paste it below.

In [ ]:
import os
# os.environ['OPENAI_API_KEY'] = 'sk-...'
assert os.environ.get('OPENAI_API_KEY'), 'set OPENAI_API_KEY first'

## 3. Wrap your OpenAI function with `@trust`

Zero configuration. `@trust` inspects your function signature and auto-detects `context` (or `reference`, `passage`, `docs`, `source`, `knowledge`, `grounding`, `retrieved`) as the grounding passage. Auto-enables the NLI contradiction signal because `styxx[nli]` is installed.

In [ ]:
from styxx import trust
import openai

client = openai.OpenAI()

@trust
def ask(question, *, context):
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[
            {'role': 'user',
             'content': f'Context: {context}\n\nQuestion: {question}\n\n'
                        f'Answer concisely using only the context.'},
        ],
        temperature=0.3,
    )
    return r.choices[0].message.content

## 4. Correct answer — passes through unchanged

The model gets a grounded question on a grounded context. `@trust` sees low novelty + no NLI contradiction → returns the response verbatim.

In [ ]:
context = '''Inception is a 2010 science fiction film written and directed
by Christopher Nolan. It stars Leonardo DiCaprio as a thief who steals
corporate secrets through dream-sharing technology.'''

ask('Who directed Inception?', context=context)

## 5. Force a hallucination — `@trust` intercepts it

Same function, but now we ask a specific question whose answer is NOT in the context. The model will confabulate a plausible number. The detector fires because the response contains high-novelty tokens (made-up dollar figure) that have no support in the reference passage.

In [ ]:
context = '''Inception is a 2010 science fiction film written and
directed by Christopher Nolan.'''

ask('What was the exact production budget of Inception, to the dollar? '
    'Reply with just the number.', context=context)

If that returned the styxx fallback message instead of a fabricated number, the detector worked.

## 6. Inspect the verdict — `on_halt='annotate'`

Return a `TrustResult` that exposes per-signal risk, halt action, and the signal table. Same interface works across OpenAI, Anthropic, local models, and anything that returns a string.

In [ ]:
@trust(on_halt='annotate')
def ask_annotated(question, *, context):
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user',
                   'content': f'Context: {context}\n\nQuestion: {question}\n\nAnswer concisely.'}],
        temperature=0.3,
    )
    return r.choices[0].message.content

result = ask_annotated(
    'What was the exact production budget of Inception, to the dollar?',
    context=context,
)

print(f'response : {result.response}')
print(f'risk     : {result.verdict.risk:.3f}')
print(f'action   : {result.verdict.action}')
print(f'halted   : {result.halted}')
print(f'attempts : {result.attempts}')
print('\nsignals:')
for s in result.verdict.signals:
    print(f'  {s.name:<22s} {s.value}')

## 7. Halt policies

`@trust(on_halt=...)` supports four policies for what happens when risk exceeds threshold:

- `fallback` (default): return a safe fallback string
- `retry`: re-call the function up to `max_retries` times; best-of-N picks the lowest-risk response
- `raise`: raise `TrustViolation` exception
- `annotate`: return `TrustResult(response, verdict)` — let the caller decide

```python
@trust(on_halt='retry', max_retries=3, threshold=0.7)
def safer_ask(q, *, context): ...
```

## 8. Honest failure modes

Styxx **does not work well** on two benchmark types, published as declared failure modes in the weights module itself:

- **Reading-comprehension extractive-span errors** (e.g. DROP-style tasks where the answer is a specific span of a passage). The wrong span is *entailed* by the passage at the NLI level; novelty signals don't fire because the tokens overlap.
- **Financial arithmetic** (e.g. FinanceBench-style calculation errors on numbers copied verbatim from the source). Novelty + NLI are semantically blind to arithmetic correctness.

Do not deploy `@trust` for production workloads in those two domains without additional domain-specific checks. Full deep-dive with the null-probe evidence: https://fathom.darkflobi.com/cognometry/failures

## Further reading

- Manifesto & methodology: https://fathom.darkflobi.com/cognometry
- Leaderboard (open submissions): https://fathom.darkflobi.com/cognometry/leaderboard
- Paper (Zenodo): https://doi.org/10.5281/zenodo.19703527
- Source: https://github.com/fathom-lab/styxx (MIT + CC-BY-4.0)